# 417. Pacific Atlantic Water Flow
**Difficulty:** 🟡 Medium · **Topic:** Graph · **LeetCode:** https://leetcode.com/problems/pacific-atlantic-water-flow/

## 💡 Concepts

**Core concept(s):** Grid **DFS/BFS**, but the clever version searches **backwards from the oceans**.

**Why it applies here:** Water flows from a cell to equal-or-lower neighbors and reaches an ocean if it can get to a border. Checking every cell forward is slow. Instead, flip it: from the ocean borders, climb to equal-or-higher cells — those are exactly the cells that can drain into that ocean. Cells reachable from *both* oceans are the answer.

**Key intuition:** Don't ask 'can this cell reach the sea?' for every cell; ask 'which cells can the sea reach if we climb uphill?' once per ocean.

---

### 📚 What is a Graph?
A **graph** is dots (**nodes/vertices**) joined by lines (**edges**). Edges can be **directed** (one-way, like prerequisites) or **undirected** (two-way, like friendships). A **grid** is just a graph where each cell links to its neighbors.
- **In Python:** usually an **adjacency list** — a `dict` mapping each node to the list of nodes it connects to.

### 📚 What is Flood Fill?
**Flood fill** is DFS/BFS on a grid: from a starting cell, spread to matching neighbors (up/down/left/right), marking them visited — like the paint-bucket tool.
- **Complexity:** **O(rows × cols)** — each cell handled once.

### 📚 What is DFS (Depth-First Search)?
**DFS** follows one path as deep as it goes, then backtracks. On graphs you must remember **visited** nodes so you don't loop forever.
- **Complexity:** **O(V + E)** — each node and edge once.
- **In Python:** recursion or an explicit stack, plus a `visited` set.

---

**Prerequisite knowledge:**
- Grid traversal (flood fill).
- Running two searches and intersecting the results.

## 📝 Problem

Water on a height grid flows to equal-or-lower neighbors. The top/left borders touch the Pacific, the bottom/right the Atlantic. Return all cells from which water can reach **both** oceans.

> Two approaches: brute (search from every cell) `O((mn)²)` and border multi-source `O(mn)`.

### Approach 1 — Search From Every Cell (worst)

**Idea:** For each cell, flow downhill and see if it can touch both a Pacific border and an Atlantic border.

**Time:** `O((mn)²)`. **Space:** `O(mn)`.

In [3]:
def pacific_atlantic_brute(heights):
    if not heights or not heights[0]:
        return []
    rows, cols = len(heights), len(heights[0])
    def reaches_both(sr, sc):              # from this cell, can water reach both oceans?
        seen = set(); stack = [(sr, sc)]; pac = atl = False
        while stack:
            x, y = stack.pop()
            if (x, y) in seen: continue
            seen.add((x, y))
            if x == 0 or y == 0: pac = True        # touched a Pacific border
            if x == rows-1 or y == cols-1: atl = True  # touched an Atlantic border
            for dx, dy in ((1,0),(-1,0),(0,1),(0,-1)):
                nx, ny = x+dx, y+dy
                if 0 <= nx < rows and 0 <= ny < cols and heights[nx][ny] <= heights[x][y]:
                    stack.append((nx, ny))  # water flows to equal-or-lower neighbors
        return pac and atl
    return [[r, c] for r in range(rows) for c in range(cols) if reaches_both(r, c)]

### Approach 2 — Climb From the Borders (optimal)

**Idea:** From every Pacific-border cell, DFS to equal-or-higher neighbors, marking cells that can drain to the Pacific. Do the same for the Atlantic. The intersection is the answer.

**Time:** `O(mn)`. **Space:** `O(mn)`.

In [ ]:
def pacific_atlantic_optimal(heights):
    if not heights or not heights[0]:
        return []
    rows, cols = len(heights), len(heights[0])
    pac, atl = set(), set()                # cells that can drain to each ocean
    def dfs(r, c, seen, prev):             # CLIMB inland from the border (uphill/level)
        if (r, c) in seen or r < 0 or c < 0 or r >= rows or c >= cols or heights[r][c] < prev:
            return
        seen.add((r, c))
        for dx, dy in ((1,0),(-1,0),(0,1),(0,-1)):
            dfs(r+dx, c+dy, seen, heights[r][c])   # move to equal-or-higher neighbors
    for c in range(cols):                  # seed from the top (Pacific) and bottom (Atlantic) rows
        dfs(0, c, pac, heights[0][c]); dfs(rows-1, c, atl, heights[rows-1][c])
    for r in range(rows):                  # seed from the left (Pacific) and right (Atlantic) columns
        dfs(r, 0, pac, heights[r][0])
        dfs(r, cols-1, atl, heights[r][cols-1])
    return [[r, c] for r in range(rows) for c in range(cols) if (r, c) in pac and (r, c) in atl]

In [5]:
# Correctness check
heights = [[1,2,2,3,5],[3,2,3,4,4],[2,4,5,3,1],[6,7,1,4,5],[5,1,1,2,4]]
a = sorted(pacific_atlantic_brute(heights))
b = sorted(pacific_atlantic_optimal(heights))
print("cells reaching both oceans:", b)
assert a == b, "approaches disagree"
assert [0,4] in b and [4,0] in b, "known cells missing"
print("\nAll tests passed")

cells reaching both oceans: [[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]

All tests passed


## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)` / `O(V+E)` | ≈ **2×** |
| `O(n log n)`      | ≈ **2×** (slightly more) |
| `O(n²)`           | ≈ **4×** |

Inputs are shaped to force the worst case while keeping recursion shallow (stars / checkerboards) so nothing overflows the stack.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    # increasing heights -> lots of reachable cells, plenty of work
    grid = [[r + c for c in range(n)] for r in range(n)]
    return (grid,)
solutions = {
    "brute   O((mn)^2)": pacific_atlantic_brute,
    "borders O(mn)    ": pacific_atlantic_optimal,
}
sizes = [8, 12, 16, 24]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Search backwards from the goal:** starting at the borders (the answers' destinations) turns `O((mn)²)` into `O(mn)`.
- **Multi-source flood fill:** seed the search from many starting cells at once.
- **Intersect two reachable sets:** cells that satisfy two conditions = overlap of two searches.
- **Signal:** "which cells can reach X and Y", "flow / spread on a grid".
- **Related problems:** Number of Islands, Surrounded Regions, Walls and Gates.
- **Common pitfalls:** (1) searching forward from every cell; (2) wrong height comparison direction when climbing vs flowing.